In [2]:
import os
import pandas as pd
import psycopg
from dotenv import load_dotenv


In [3]:
load_dotenv()

DB_NAME = os.getenv("POSTGRES_DB")
DB_USER = os.getenv("POSTGRES_USER")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_HOST = "localhost"
DB_PORT = os.getenv("POSTGRES_PORT")

In [4]:
print("Database:", DB_NAME)
print("User:", DB_USER)
print("Host:", DB_HOST)
print("Port:", DB_PORT)
print("Password loaded:", DB_PASSWORD is not None)

Database: complex_solutions
User: complex_admin
Host: localhost
Port: 5433
Password loaded: True


In [5]:
connection = psycopg.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
)

print("Conexión a PostgreSQL establecida correctamente.")

Conexión a PostgreSQL establecida correctamente.


In [6]:
with connection.cursor() as cursor:
    cursor.execute("SELECT current_database(), current_user;")
    result = cursor.fetchone()

print(result)

('complex_solutions', 'complex_admin')


In [7]:
with connection.cursor() as cursor:
    cursor.execute("""
        SELECT COUNT(*)
        FROM customers;
    """)
    
    total_customers = cursor.fetchone()[0]

print("Total de clientes:", total_customers)

Total de clientes: 362


In [8]:
query = """
SELECT *
FROM customers;
"""

customers_df = pd.read_sql(query, connection)

customers_df.head()

/var/folders/b2/pplc2ntj7c327kj3hlb_ndw40000gn/T/ipykernel_20780/2101952970.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customers_df = pd.read_sql(query, connection)


,customer_id,customer_type,company_name,customer_segment,city,province,signup_date,acquisition_channel,sales_rep_id
0,1,B2B,Customer 00001,SMB,Madrid,Madrid,2026-09-08,OUTBOUND_SALES,1.0
1,2,B2C,Customer 00002,CONSUMER,Barcelona,Barcelona,2020-09-17,ONLINE,NaN
2,3,B2C,Customer 00003,CONSUMER,Valencia,Valencia,2024-04-25,ONLINE,NaN
3,4,B2B,Customer 00004,SMB,Sevilla,Sevilla,2024-04-12,OUTBOUND_SALES,1.0
4,5,B2C,Customer 00005,CONSUMER,Bilbao,Bizkaia,2026-09-06,ONLINE,NaN


In [9]:
import sqlalchemy

In [10]:
from sqlalchemy import create_engine

engine = create_engine(
    f"postgresql+psycopg://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Engine de SQLAlchemy creado correctamente.")

Engine de SQLAlchemy creado correctamente.


In [11]:
query = """
SELECT *
FROM customers;
"""

customers_df = pd.read_sql(query, engine)

customers_df.head()

,customer_id,customer_type,company_name,customer_segment,city,province,signup_date,acquisition_channel,sales_rep_id
0,1,B2B,Customer 00001,SMB,Madrid,Madrid,2026-09-08,OUTBOUND_SALES,1.0
1,2,B2C,Customer 00002,CONSUMER,Barcelona,Barcelona,2020-09-17,ONLINE,NaN
2,3,B2C,Customer 00003,CONSUMER,Valencia,Valencia,2024-04-25,ONLINE,NaN
3,4,B2B,Customer 00004,SMB,Sevilla,Sevilla,2024-04-12,OUTBOUND_SALES,1.0
4,5,B2C,Customer 00005,CONSUMER,Bilbao,Bizkaia,2026-09-06,ONLINE,NaN


In [12]:
customers_df.shape

(362, 9)

In [13]:
customers_df.columns

Index(['customer_id', 'customer_type', 'company_name', 'customer_segment',
       'city', 'province', 'signup_date', 'acquisition_channel',
       'sales_rep_id'],
      dtype='str')

In [14]:
customers_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 362 entries, 0 to 361
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          362 non-null    int64  
 1   customer_type        362 non-null    str    
 2   company_name         362 non-null    str    
 3   customer_segment     362 non-null    str    
 4   city                 362 non-null    str    
 5   province             362 non-null    str    
 6   signup_date          362 non-null    object 
 7   acquisition_channel  362 non-null    str    
 8   sales_rep_id         122 non-null    float64
dtypes: float64(1), int64(1), object(1), str(6)
memory usage: 25.6+ KB


In [15]:
customers_df.isna().sum()


customer_id              0
customer_type            0
company_name             0
customer_segment         0
city                     0
province                 0
signup_date              0
acquisition_channel      0
sales_rep_id           240
dtype: int64

In [16]:
customers_df[customers_df["sales_rep_id"].isna()]["customer_type"].value_counts()

customer_type
B2C    240
Name: count, dtype: int64

In [17]:
customers_df[
    (customers_df["customer_type"] == "B2B") &
    (customers_df["sales_rep_id"].isna())
]


,customer_id,customer_type,company_name,customer_segment,city,province,signup_date,acquisition_channel,sales_rep_id


In [18]:
customers_df[
    customers_df["customer_type"] == "B2C"
]["sales_rep_id"].notna().sum()

np.int64(0)

In [19]:
customers_df["customer_type"].value_counts()

customer_type
B2C    240
B2B    122
Name: count, dtype: int64

In [22]:
customers_df["customer_type"].value_counts(normalize=True)

customer_type
B2C    0.662983
B2B    0.337017
Name: proportion, dtype: float64

In [23]:
customers_df["acquisition_channel"].value_counts()

acquisition_channel
ONLINE            241
OUTBOUND_SALES    121
Name: count, dtype: int64

In [24]:
pd.crosstab(
    customers_df["customer_type"],
    customers_df["acquisition_channel"]
)

acquisition_channel,ONLINE,OUTBOUND_SALES
customer_type,,
B2B,1,121
B2C,240,0
